# TF-IDF Based Search / Recommendation System

## 1. Import Required Libraries

In [1]:
import pandas as pd
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Download NLTK Resources

These resources are required for stopword removal and lemmatization.

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 3. Create a Dataset

The dataset contains 10 short movie/document descriptions. Each document has a title and a description.

In [3]:
data = [
    {
        "title": "Space Robot Adventure",
        "description": "A young astronaut travels through space with intelligent robots to save a lost planet."
    },
    {
        "title": "Healthy Life",
        "description": "A documentary about healthy food, daily exercise, and balanced lifestyle habits."
    },
    {
        "title": "Ocean Mystery",
        "description": "A group of scientists explores the deep ocean and discovers a hidden underwater world."
    },
    {
        "title": "Robot City",
        "description": "In a futuristic city, robots and humans work together to solve dangerous problems."
    },
    {
        "title": "Cooking Fresh",
        "description": "A chef teaches how to prepare fresh vegetables, healthy meals, and simple home recipes."
    },
    {
        "title": "Galaxy Wars",
        "description": "A space adventure where brave pilots fight aliens across different galaxies."
    },
    {
        "title": "Mountain Survival",
        "description": "A climber must survive extreme weather conditions on a dangerous mountain."
    },
    {
        "title": "AI Future",
        "description": "This film explains artificial intelligence, machine learning, and the future of smart robots."
    },
    {
        "title": "Farm to Table",
        "description": "Farmers grow organic vegetables and fruits to support healthy eating."
    },
    {
        "title": "Time Machine",
        "description": "A scientist builds a machine to travel through time and change history."
    }
]

df = pd.DataFrame(data)
df

,title,description
0,Space Robot Adventure,A young astronaut travels through space with i...
1,Healthy Life,"A documentary about healthy food, daily exerci..."
2,Ocean Mystery,A group of scientists explores the deep ocean ...
3,Robot City,"In a futuristic city, robots and humans work t..."
4,Cooking Fresh,A chef teaches how to prepare fresh vegetables...
5,Galaxy Wars,A space adventure where brave pilots fight ali...
6,Mountain Survival,A climber must survive extreme weather conditi...
7,AI Future,"This film explains artificial intelligence, ma..."
8,Farm to Table,Farmers grow organic vegetables and fruits to ...
9,Time Machine,A scientist builds a machine to travel through...


## 4. Preprocess the Text

The preprocessing function performs the following steps:

1. Converts text to lowercase
2. Removes punctuation
3. Splits text into words
4. Removes stopwords
5. Lemmatizes each word

In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenize text
    words = text.split()

    # Remove stopwords and lemmatize
    cleaned_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(cleaned_words)

## 5. Apply Preprocessing to Documents

In [5]:
df["clean_description"] = df["description"].apply(preprocess_text)
df[["title", "description", "clean_description"]]

,title,description,clean_description
0,Space Robot Adventure,A young astronaut travels through space with i...,young astronaut travel space intelligent robot...
1,Healthy Life,"A documentary about healthy food, daily exerci...",documentary healthy food daily exercise balanc...
2,Ocean Mystery,A group of scientists explores the deep ocean ...,group scientist explores deep ocean discovers ...
3,Robot City,"In a futuristic city, robots and humans work t...",futuristic city robot human work together solv...
4,Cooking Fresh,A chef teaches how to prepare fresh vegetables...,chef teach prepare fresh vegetable healthy mea...
5,Galaxy Wars,A space adventure where brave pilots fight ali...,space adventure brave pilot fight alien across...
6,Mountain Survival,A climber must survive extreme weather conditi...,climber must survive extreme weather condition...
7,AI Future,"This film explains artificial intelligence, ma...",film explains artificial intelligence machine ...
8,Farm to Table,Farmers grow organic vegetables and fruits to ...,farmer grow organic vegetable fruit support he...
9,Time Machine,A scientist builds a machine to travel through...,scientist build machine travel time change his...


## 6. Convert Documents into TF-IDF Vectors

TF-IDF gives higher importance to words that are important in one document but not common across all documents.

In [6]:
vectorizer = TfidfVectorizer()

document_vectors = vectorizer.fit_transform(df["clean_description"])

print("TF-IDF Matrix Shape:", document_vectors.shape)
print("Number of Documents:", document_vectors.shape[0])
print("Number of Unique Terms:", document_vectors.shape[1]
)

TF-IDF Matrix Shape: (10, 76)
Number of Documents: 10
Number of Unique Terms: 76


## 7. Accept and Preprocess User Query

In [7]:
user_query = "space adventure with robots"

clean_query = preprocess_text(user_query)

print("Original Query:", user_query)
print("Cleaned Query:", clean_query)

Original Query: space adventure with robots
Cleaned Query: space adventure robot


## 8. Calculate Cosine Similarity

Cosine similarity compares the query vector with every document vector.

A higher score means the document is more similar to the query.

In [8]:
query_vector = vectorizer.transform([clean_query])

similarity_scores = cosine_similarity(query_vector, document_vectors).flatten()

df["similarity_score"] = similarity_scores

df[["title", "similarity_score"]].sort_values(by="similarity_score", ascending=False)

,title,similarity_score
5,Galaxy Wars,0.386641
0,Space Robot Adventure,0.299027
7,AI Future,0.127456
3,Robot City,0.127456
1,Healthy Life,0.000000
2,Ocean Mystery,0.000000
4,Cooking Fresh,0.000000
6,Mountain Survival,0.000000
8,Farm to Table,0.000000
9,Time Machine,0.000000


## 9. Return the Top 3 Results

In [10]:
top_3_results = df.sort_values(by="similarity_score", ascending=False).head(3)

print("User Query:", user_query)
print("\nTop 3 Recommended Documents:")

for index, row in top_3_results.iterrows():
    print("Title:", row["title"])
    print("Description:", row["description"])
    print("Similarity Score:", round(row["similarity_score"], 4))
    print("-" * 60)

User Query: space adventure with robots

Top 3 Recommended Documents:
Title: Galaxy Wars
Description: A space adventure where brave pilots fight aliens across different galaxies.
Similarity Score: 0.3866
------------------------------------------------------------
Title: Space Robot Adventure
Description: A young astronaut travels through space with intelligent robots to save a lost planet.
Similarity Score: 0.299
------------------------------------------------------------
Title: AI Future
Description: This film explains artificial intelligence, machine learning, and the future of smart robots.
Similarity Score: 0.1275
------------------------------------------------------------


## 10. Create a Reusable Search Function

This function allows us to search using any query and return the top 3 results.

In [11]:
def search_documents(query, top_n=3):
    clean_query = preprocess_text(query)
    query_vector = vectorizer.transform([clean_query])
    similarity_scores = cosine_similarity(query_vector, document_vectors).flatten()

    results = df.copy()
    results["similarity_score"] = similarity_scores
    results = results.sort_values(by="similarity_score", ascending=False).head(top_n)

    return results[["title", "description", "similarity_score"]]

## 11. Test the Search System

Example query: **healthy food**

In [12]:
search_documents("healthy food", top_n=3)

,title,description,similarity_score
1,Healthy Life,"A documentary about healthy food, daily exerci...",0.453462
8,Farm to Table,Farmers grow organic vegetables and fruits to ...,0.164545
4,Cooking Fresh,A chef teaches how to prepare fresh vegetables...,0.145730


## 12. Another Example Query

Example query: **space adventure with robots**

In [13]:
search_documents("space adventure with robots", top_n=3)

,title,description,similarity_score
5,Galaxy Wars,A space adventure where brave pilots fight ali...,0.386641
0,Space Robot Adventure,A young astronaut travels through space with i...,0.299027
7,AI Future,"This film explains artificial intelligence, ma...",0.127456
